## Stateless Logic Circuits --> Stateful Logic Circuits

- Up to this point, we have covered all **stateless operations** performed by a computer

- These are useful to a point, but we are still missing some important bits of a modern computer
    - Modern computers needs persistent memory, and stateless operations cannot provide this
    - We have seen have stateless operations handle inputs. But what exactly coordinates the input and output? 
    
- These 2 questions lead us to our next section in building up a running computer from scratch. To execute software, a computer needs:
    - **Memory**: To retain state across instructions 
    - **Clock**: To step through execution sequentially

- In this notebook, we will construct stateful operations from scratch!
    - **SR Latch & D Latch:** Primitive 1-bit memory cells built using feedback loops in NOR/NAND logic gates.
    - **D Flip-Flops:** Edge-triggered memory elements that only update state on clock transitions.
    - **Registers:** Parallel collections of flip-flops used to store multi-bit data words (e.g., 4-bit CPU registers).
    - **Register File:** An addressable array of registers with two read ports, built using Decoders and MUXes from Notebook 3.
    - **Clock Driver:** Runs each clock cycle as CLK = 0 then CLK = 1, and holds every flip-flop's state between cycles.
    - **Data RAM:** A 16-word memory for program data, built the same way as the register file.
    - **ROM:** Read-only memory that holds the program.
    - **Tests:** Checks every component against a plain Python model, cycle by cycle.

In [ ]:
from utils import *

### 1. Latches (1-Bit Bistable Memory)

#### Set Reset Latch

- Let's start by seeing how stateless operations, when combined in a a specific configuration, can let us retain the state of a bit!

- We start by illustrating the basic concept:
    - Instead of seeing "memory" as some magical state-retaining machine, the main insight is that persistent state can be obtained by feeding a sequence of stateless operations inputs in a loop!

- We'll start with the most basic example, the Set-Reset Latch (SR Latch) to illustrate this point

In [ ]:
from utils import *

def sr_latch(set: TRANSISTOR_OUTPUT, reset: TRANSISTOR_OUTPUT, val_prev: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    The purpose of an SR Latch is to allow us to persist the state of 1 bit of information. Let's call this `val`.
    
    To do this, we take in 3 inputs:
        - A `set` signal, indicating whether we should set `val` to 1
        - A `reset` signal, indicating whether we should set `val` to 0
        - The previous value held by `val`, which call `val_prev`
        - Note that in an SR Latch, `set` and `reset` must not both be 1 at the same time. 
    
    Using these, the SR latch outputs a value of `val`, and its inverse `val_prime`
        - Note that though `val_prime` is the inverse of `val`, it is not derived by passing `val` through a NOT gate
        - Instead, `val_prime` is computed separately, to ensure that there is no delay between the output `val` and `val_prime` being emitted
    
    The truth table below shows how the SR Latch turns the inputs into persistent byte value storage.
        - However, keep in mind that this doesn't always happen in one pass!
        - Sometimes, we can end up with intermediate val/val_prime values that require a few steps for its value to settle
        - We will show these in the truth table below

    Truth Table / Trace:
        - val_prev = 0 (val_prime_prev = 1):

            - set = 0, reset = 0 (HOLD STATE): Neither set nor reset. We want the circuit to output the same `val` it already has
                - Pass 1: val_prime = NOR(set=0, val_prev=0) = 1
                - Pass 1: val = NOR(reset=0, val_prime=1) = 0
                - Settled: (val=0, val_prime=1)

            - set = 0, reset = 1 (RESET STATE): Reset state. But since val = 0 already, we continue to output val = 0
                - Pass 1: val_prime = NOR(set=0, val_prev=0) = 1
                - Pass 1: val = NOR(reset=1, val_prime=1) = 0
                - Settled: (val=0, val_prime=1)

            - set = 1, reset = 0 (SET STATE): Set state.
                - Pass 1: val_prime = NOR(set=1, val_prev=0) = 0
                - Pass 1: val = NOR(reset=0, val_prime=0) = 1
                - +++++++++++++++++++++++++++++++++++++++++++++++++
                - Pass 2: val_prime = NOR(set=1, val=1) = 0
                - Pass 2: val = NOR(reset=0, val_prime=0) = 1
                - +++++++++++++++++++++++++++++++++++++++++++++++++                
                - Settled: (val=1, val_prime=0)

        - val_prev = 1 (val_prime_prev = 0):

            - set = 0, reset = 0 (HOLD STATE):
                - Pass 1: val_prime = NOR(set=0, val_prev=1) = 0
                - Pass 1: val = NOR(reset=0, val_prime=0) = 1
                - Settled: (val=1, val_prime=0)

            - set = 0, reset = 1 (RESET STATE):
                - Pass 1: val_prime = NOR(set=0, val_prev=1) = 0
                - Pass 1: val = NOR(reset=1, val_prime=0) = 0
                - +++++++++++++++++++++++++++++++++++++++++++++++++
                - Pass 2: val_prime = NOR(set=0, val=0) = 1
                - Pass 2: val = NOR(reset=1, val_prime=1) = 0
                - +++++++++++++++++++++++++++++++++++++++++++++++++
                - Settled: (val=0, val_prime=1)

            - set = 1, reset = 0 (SET STATE):
                - Pass 1: val_prime = NOR(set=1, val_prev=1) = 0
                - Pass 1: val = NOR(reset=0, val_prime=0) = 1
                - Settled: (val=1, val_prime=0)
    '''
    if set == 1 and reset == 1:
        raise ValueError("Invalid State: 'set' and 'reset' cannot both be 1.")

    val = val_prev
    val_prime = 0 if val_prev == 1 else 1

    # Loop until the cross-coupled NOR gates stabilize into equilibrium
    prev_state = None
    while (val, val_prime) != prev_state:
        prev_state = (val, val_prime)

        # Gate 1: Drives val_prime using external 'set' and internal feedback 'val'
        val_prime = cmos_NOR(set, val)

        # Gate 2: Drives val using external 'reset' and internal feedback 'val_prime'
        val = cmos_NOR(reset, val_prime)

    return val, val_prime


#### D Latch

- The SR latch works, but notice its very annoying flaw. The moment you pass both set=1 and reset=1, the entire SR latch breaks. We simply cannot have this set of inputs
    - In a circuit board, there are many reasons why this sometimes happens. External interference, electrical leaks etc can all add up to reduce reliability
    - This means that any board using an SR latch is always 1 moment away from disaster. We never know when we may accidentally send (1,1) to the SR latch and cause a meltdown

- THe D Latch was invented to solve precisely this problem

In [ ]:
def d_latch(setval: TRANSISTOR_OUTPUT, enable: TRANSISTOR_OUTPUT, val_prev: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    Similar to SR Latch, the D latch is designed to handle the persistence of 1 bit of data. 
    In fact, the D latch LITERALLY contains an SR Latch, just with better UI to avoid mistakes with the inputs.

    Inputs   
        - `setval`
            - Unlike the SR latch, which uses 1 input for `setval` and 1 for `reset`, the D Latch only takes 1 input for both. 
            - If the input is 1, it signals `setval`. Else, it signals `reset`
            - This avoids the possibility of having both setval=1 and reset=1, which was the problem with the SR latch setup
        - `enable`
            - An extra gate that determines whether the `setval` input is passed foward
            - If `enable = 0`, `setval` is always ignored, no matter its value
        - `val_prev`
            - `val` from previous time step
    '''
    reset = cmos_NOT(setval)
    
    setval = cmos_AND(setval, enable)
    reset = cmos_AND(reset, enable)
    
    return sr_latch(setval, reset, val_prev)

##### Applications of D-Latch

- The feedback loop inside a latch shows up in both logic processing and in memory
    - In logic processing, D-Latches (and the flip-flops built from them in section 2) hold values between pipeline stages and inside CPU registers
    - In memory, Static RAM (SRAM), used for high-speed CPU caches, stores each bit in the same kind of feedback loop. But an SRAM cell is not a D-Latch; it is a stripped-down 6-transistor circuit (details below)
    - Main memory uses Dynamic RAM (DRAM) instead, which stores each bit as charge in a tiny capacitor that leaks and must be refreshed. Persistent storage (Flash/SSDs) works differently again

- We will look at both use cases to see where the latch idea shows up

##### D Latch in Logic Processing

- In computational logic a D-Latch acts as a timed firewall and shield for chained pipelines (e.g. I want to connect an Adder to a Multiplier):
    - **Shielding Intermediate Noise:** 
        - In a computer, all computations take a step when the CPU sends a tick signal
        - In between tick signals, values can fluctuate as the current flows through the circuit
        - Suppose we want to compose an ADDER with a MULTIPLIER. This fluctuating signal can cause the ADDER's output to fluctuate, which causes the MULTIPLIER's output to fluctutate
        - Downstream circuits may capture these values erroneously, which affects their processing
        - By placing a D-Latch between the ADDER and MULTIPLIER, and passing the CPU tick as a `enable` signal to the D-Latch, no value is captured by the D-Latch until the signal has settled. 
        - Therefore, the multiplier never receives garbaage input
    - **Pipeline Isolation:** 
        - The D Latch holds the settled result steady, so the next processing stage MULTIPLIER works cleanly on the current data without noise, while the ADDER can  immediately begin computing the next set of inputs in parallel

- Recall from our function above that a D Latch takes in 3 inputs; `enable`, `set`, and `val_prev`. So in logic processing: 
    - `enable` takes its value from the system clock wire, which ensure that the D Latch only stores the value when the system clock actually ticks, and so it never accidentally stores intermediate noise 
    - `set` takes in value from its own dedicated wire connected to the upstream operator (e.g. `ADDER`)
    - `val_prev` is the value that's already in the D Latch

##### Feedback Loops in Cache Memory (SRAM Grid Arrays)

- In memory architecture, bit cells are arranged into a 2D grid to form **Static RAM (SRAM)**, the fast memory used for CPU caches ($L1, L2, L3$).

- Main system RAM uses **Dynamic RAM (DRAM)**, which stores each bit as charge in a microscopic capacitor that must be refreshed periodically. SRAM keeps each bit in a transistor feedback loop, so it needs no refresh and is faster, but takes more silicon per bit.
	- Note: While we model memory using gate-level flip-flops for logic simulation, real CPU caches use compact 6-transistor (6T) SRAM cells that rely on physical transistor feedback to achieve ultra-fast, high-density storage."

- **The Grid Architecture (Word Lines & Bit Lines):**
    - **Word Line (Horizontal):** Connected to the access transistors of every cell in that row. Driven by the Address Decoder to connect an entire row to the bit lines at once.
    - **Bit Line Pair (Vertical):** $BL$ and $\overline{BL}$ run down each column, and carry the bit into or out of every cell in that column.

- **Write:** The write drivers force $BL$ to the new bit and $\overline{BL}$ to its inverse, then the word line goes high. The drivers are stronger than the cell's small NOT gates, so they overpower the loop and flip it.

- **Read:** Both bit lines are first charged to 1 (precharge). Then the word line goes high. The side of the cell holding 0 pulls its bit line down slightly, and a **sense amplifier** at the bottom of the column detects which line dropped.

- **Column Masking:**
    - A word line opens every cell in its row at once, but a write may only target some columns (e.g. one byte).
    - The targeted columns get driven bit lines. The other columns keep their bit lines precharged, which acts like a read, so those cells keep their value.

- In our simulation, we model memory with the flip-flop registers from sections 2 and 3, because we are working at the logic-gate level. The addressing is the same: a decoder picks the row, a MUX picks the output.

- *(We will explore the complete multi-row layout and address decoding logic in Section 4 on `Register File` and Section 6 on `Data RAM`).*

### 2. Edge-Triggered D Flip-Flop

- We have seen how D Latches work. Let's zoom in on the portion where we said that D Latches sit between operators, wired to the computer's clock signal line 
    - Suppose this pipeline: `ADDER --> D Latch --> MULTIPLIER`
    - We said that by wiring D Latch to the computer's clock, it only accepts and stores the value from ADDER when the clock ticks
    - This prevents the D Latch from storing intermediate noise

- Notice that this is only one half of the problem. Yes, we have prevented noise from ADDER from hitting D Latch until the next clock step. But once D Latch stores a value, it IMMEDIATELY flows to MULTIPLIER, no matter what multiplier is meant to do. 
    - What we want: 
        - t0: adder operation 
        - t1: D Latch accepts and stores value A, Multiplier working on previous D Latch value. Adder starts working on new value B
        - t2: D Latch passes new value A onward to Multiplier, and accepts new value B from ADDER
    - What we get with a pure D Latch
        - t0: adder operation 
        - t1: D Latch accepts and stores new value A, Multiplier imemdiately gets new value A
        - t2: D Latch accepts and stores new value B, Multiplier imemdiately gets new value B
    - Multiplier never has a "clock tick" of time to work on new value A!

- This is why a pure D Latch is not enough. We need something that sits between ADDER and D Latch, but we also need something that sits between D Latch and MULTIPLIER!
    - Basically, we need a mechanism to ensure that we only ever advance 1 operation per clock tick

- This is where an **Edge-Triggered D Flip Flop** comes in


#### How does it work?

- The idea is quite simple. Instead of a pure D Latch, we combine 2 D latches back to back!
    - The clock wire is still connected to both D latches, but one of them has a NOT modifier
    - When clock = 0: 
        - The first (master) latch is enabled, and it takes in the value from its upstream operator
        - The second (slave) latch is disabled, so nothing can reach it. It stably holds the value it was provided and passes it to the downstream operator
    - When clock = 1:
        - The first (master) latch is disabled, and it blocks any new value from input D
        - The second (slave) latch is enabled, and takes in the value from the master D latch, and passes it downstream

```
                            D FLIP-FLOP
           +-------------------------------------------+
           |                                           |
           |   +---------------+   +---------------+   |
Input D --->---| Master Latch  |---|  Slave Latch  |---|---> Output Q
           |   |   (E = CLK')  |   |   (E = CLK)   |   |
           |   +---------------+   +---------------+   |
           +-------------------------------------------+
                     ^                   ^
                     |      +-----+      |
CLK -----------------+------| NOT |------+
                            +-----+
```

In [ ]:
def d_flip_flop(input: TRANSISTOR_OUTPUT, enable: TRANSISTOR_OUTPUT, slave_val_prev: TRANSISTOR_OUTPUT, master_val_prev: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    Rising-Edge Triggered Master-Slave D Flip-Flop.
    
    Inputs:
        - input Input data bit
        - enable: From system clock signal
        - slave_val_prev: Stored output Q (from Slave)
        - master_val_prev: Stored intermediate state Q (from Master)
        
    Returns:
        - (Q, Q_bar, master_Q): Updated output Q, inverse Q_bar, and new internal Master Q state
    '''
    not_enable = cmos_NOT(enable)
    
    # Stage 1: Master Latch (Enabled when Clock is LOW)
    master_q, _ = d_latch(input, not_enable, master_val_prev)
    
    # Stage 2: Slave Latch (Enabled when Clock is HIGH)
    slave_q, not_slave_q = d_latch(master_q, enable, slave_val_prev)
    
    return slave_q, not_slave_q, master_q

### 3. N-Bit Registers

- We have seen how a single **Edge-Triggered D Flip-Flop** acts as a 1-bit memory airlock for CPU pipelines.

- In modern computing, CPUs operate on multi-bit chunks of data simultaneously (e.g., 8-bit, 32-bit, or 64-bit words). A **Register** is simply an array of $N$ D Flip-Flops connected in parallel, sharing a common clock and control signals.

#### Building an N-Bit Register

- To build an 8-bit register, we line up 8 individual D Flip-Flops side-by-side.

- **Data Bus ($D_0 \dots D_7$):** Each flip-flop receives 1 bit from the 8-bit input bus
    - This is equivalent to the `input` field that each flip flop receives from before

- **Clock Line ($CLK$):** A single system clock wire is routed to the `enable` pin of **all 8 flip-flops simultaneously**. On the rising edge ($0 \rightarrow 1$), all 8 bits are captured in exact synchronization.
    - This is equivalent to the `enable` field that each flip flop receives from before

```
                                     8-BIT REGISTER
        +-----------------------------------------------------------------------+
        |                                                                       |
Data    |   D[0]           D[1]           D[2]                  D[7]            |    Outputs
Bus ====|====|==============|==============|=====================|==============|===> Q[0..7]
D[0..7] |    |              |              |                     |              |
        |    v              v              v                     v              |
        |  +---+          +---+          +---+                 +---+            |
        |  | D |          | D |          | D |                 | D |            |
        |  |   |--> Q[0]  |   |--> Q[1]  |   |--> Q[2]  ...    |   |--> Q[7]    |
        |  |   |          |   |          |   |                 |   |            |
        |  |DFF0          |DFF1          |DFF2                 |DFF7            |
        |  +---+          +---+          +---+                 +---+            |
        |    ^              ^              ^                     ^              |
        |    |              |              |                     |              |
        +----|--------------|--------------|---------------------|--------------+
             |              |              |                     |
CLK ---------+--------------+--------------+---------------------+
             (Global Clock Line connected to all 8 Enable pins)

```

- From the schematic above, we see that instead of a single `CLK` input, we now have 8 of the same inputs from `CLK` to the 8 flip flops

- The data bus passes one bit of data per flip flop, which serves its job of accepting new data with every clock tick

##### Complication: What if operations take longer than 1 clock tick?

- Notice one problem here: we have no way to control when a flipflop should or should not accept data from the data bus! Once the clock ticks, the over in the flip flop is overwritten, no matter what

- This works if all downstream operations can complete within 1 clock tick. However, there are definitely operations that need more than 1 clock tick to complete!
    - If we stick to the previous schematic, those operations would be impossible, because they will be interrupted with every tick of the system clock signal

- To get around this, a register needs something to give it finer control of every flip flop; a bit that tells it whether or not the downstream operation has completed. This is known as the **LOAD** line
    - This goes beyond the `enable` signal from the clock line
    - The clock line the register that it CAN write the next operation's value, the load line tells that register that it SHOULD write the next operation's value

- Think of it as an **AND** operation. We need both clock `enable` and load `enable` before the register records the value

- This is a schematic for one such flip flop that takes in both the load and clock lines
    - Every D flipflop is paired with a `MUX`, which takes in the flip flop's existing value, and a data bus value
	- Depending on the LOAD signal, it outputs one to the other to the flip flop

```text
                       +---------------+
New Data Bus --------> | 1             |
                       |     MUX       |-----> [ D Flip-Flop ]
Existing Output (Q) -> | 0             |              ^
                       +---------------+              |
                               ^                      |
                               |                      |
LOAD Signal -------------------+                      |
                                                      |
System Clock (CLK) -----------------------------------+
```

- Extending this to a register of 8 bits:

```text
									   8-BIT REGISTER WITH LOAD ENABLE
        +-----------------------------------------------------------------------------------------------+
        |                                                                                               |
Data    |   D[0]               D[1]                               D[7]                                  |   Outputs
Bus ====|====|==================|==================================|====================================|==> Q[0..7]
D[0..7] |    |  +--------+      |  +--------+                      |  +--------+                        |
        |    |  |        |      |  |        |                      |  |        |                        |
        |    +->|1       |      +->|1       |                      +->|1       |                        |
        |       |  MUX0  |--+      |  MUX1  |--+                      |  MUX7  |--+                     |
        |  +--->|0       |  |  +-->|0       |  |                 +--->|0       |  |                     |
        |  |    +--------+  |  |   +--------+  |                 |    +--------+  |                     |
        |  |        ^       |  |       ^       |                 |        ^       |                     |
        |  |        | (L)   |  |       | (L)   |                 |        | (L)   |                     |
LOAD ---|--|--------+-------|---|------+-------|-----------------|--------+-------|---------------------+
        |  |                v   |              v                 |                v                     |
        |  |              +---+ |            +---+               |              +---+                   |
        |  |              | D | |            | D |               |              | D |                   |
        |  +--------------|   |<+------------|   |<--------------+--------------|   |<---+              |
        |  (Q0)           |DFF0| (Q1)        |DFF1|               (Q7)          |DFF7|   |              |
        |                 +---+              +---+                              +---+    |              |
        |                   ^                  ^                                  ^      |              |
        |                   | (C)              | (C)                              | (C)  |              |
CLK ----|-------------------+------------------+----------------------------------+------|--------------+
        |                                                                                |
        +--------------------------------------------------------------------------------+
```

#### Python Implementation

In [ ]:
def n_bit_register(
    inputs: list[TRANSISTOR_OUTPUT], 
    enable: TRANSISTOR_OUTPUT, 
    load: TRANSISTOR_OUTPUT, 
    slave_vals_prev: list[TRANSISTOR_OUTPUT], 
    master_vals_prev: list[TRANSISTOR_OUTPUT]
) -> tuple[list[TRANSISTOR_OUTPUT], list[TRANSISTOR_OUTPUT], list[TRANSISTOR_OUTPUT]]:
    '''
    N-Bit Synchronous Register with Load Enable using Master-Slave D Flip-Flops.
    
    Inputs:
        - inputs: List of N input data bits
        - enable: System clock signal
        - load: Control signal (1 = load new input on tick, 0 = hold current Q)
        - slave_vals_prev: List of N stored output bits (from Slaves)
        - master_vals_prev: List of N stored intermediate bits (from Masters)
        
    Returns:
        - (outputs, not_outputs, new_master_states): Tuple of outputs Q, ~Q, and internal Master states
    '''
    n = len(inputs)
    outputs = [0] * n
    not_outputs = [0] * n
    new_master_states = [0] * n
    
    for i in range(n):
        # MUX selects between external input bus (load=1) and recirculating Q (load=0)
        d_in = mux_2to1(d0=slave_vals_prev[i], d1=inputs[i], sel=load)
        
        # Execute Master-Slave D Flip-Flop
        slave_q, not_slave_q, master_q = d_flip_flop(
            input=d_in, 
            enable=enable, 
            slave_val_prev=slave_vals_prev[i], 
            master_val_prev=master_vals_prev[i]
        )
        
        outputs[i] = slave_q
        not_outputs[i] = not_slave_q
        new_master_states[i] = master_q
        
    return outputs, not_outputs, new_master_states

### 4. Register File (Addressable Registers)

#### Basics

- We now know how to build an **N-Bit Register** that stores a word of data safely using a `LOAD` signal.

- However, CPUs contain multiple registers (e.g., $R_0, R_1, R_2, R_3$). To manipulate specific registers without running individual control lines to every single one, we group them into an **Addressable Memory Array** 
	- This is also known as a **Register File**
	- Grouping registers into a unified array "file" allows the system to read from or write to specific registers using binary addresses, rather than running individual control and data lines to every single register on the chip.

- Register files appear across almost every major functional unit in a computer, because every part of a computer will need to be able to hold on to memory while operataions take place
	- CPU Core & ALU Data Path: Holds general-purpose working registers ($R_0 \dots R_n$) that feed operands directly into the Arithmetic Logic Unit (ALU) and store execution results. 
	- Execution Pipelines: Serves as pipeline registers between stages (Fetch, Decode, Execute, Memory, Writeback) to hold intermediate instruction metadata
	- GPU Compute Units: Contains massive vector register files that maintain state for thousands of concurrent execution threads
	- Control Units & Peripherals: Stores system control flags, program status registers, and memory-mapped configuration registers for hardware devices

- Not all of these Registers will be built the same way, though they will share the same idea. In this notebook, we build a register specifically for the ALU, since we will be building up to a working CPU
	- A simple pipeline buffer or peripheral register file might only need 1 read port to read one value at a time
	- However, the ALU is used to work on many operations that require transformation of multiple values in some way (e.g. `ADD R1, R2`)
	- So if the register file on the ALU only had 1 read port, the CPU would need to spend two clock cycles fetching $R_1$ and $R_2$ one after another, before the ALU could perform the addition
	- As such, in our implementation, we will have 2 read ports on our register. This is just to highlight that there is no universality in the design for registers; you can design according to what you need

```text
				REGISTER FILE (4 registers x N bits, Dual Read Ports)

          +------------------------------------------------------------------------------------+
          |                                                                                    |
Write     |  D[0..N-1] (shared write data bus)                                                 |
Data  ====|===========================+                                                        |
          |                           | (to D input of R0..R3)                                 |
          |                           v                                                        |
          |  +----------+    +-----------------+                                               |
Write     |  |          |-W0>| Register 0 (R0) |== Q0 ==+                                      |
Address --|->|  2-to-4  |-W1>| Register 1 (R1) |== Q1 ==|+                                     |
[0..1]    |  | decoder  |-W2>| Register 2 (R2) |== Q2 ==||+                                    |
          |  |          |-W3>| Register 3 (R3) |== Q3 ==|||+                                   |
          |  +----------+    +-----------------+        ||||                                   |
          |       ^                   ^                 ||||                                   |
LOAD -----|-------+ (decoder enable)  |                 ||||                                   |
          |                           |                 ||||   +--------------------+          |
Read      |                           |                 ||||==>|  4-to-1 N-bit MUX  |=========>|==> Read Data A
Addr A ---|---------------------------|-----------------|||--->|      (Port A)      |          |
          |                           |                 ||||   +--------------------+          |
          |                           |                 ||||                                   |
Read      |                           |                 ||||   +--------------------+          |
Addr B ---|---------------------------|-----------------|||--->|  4-to-1 N-bit MUX  |=========>|==> Read Data B
          |                           |                 ||||==>|      (Port B)      |          |
CLK ------|---------------------------+                 ||||   +--------------------+          |
          |                    (to all R0..R3 flip-flops)||||                                   |
          |                                             (Q0..Q3 shared bus tapped by both MUXes)|
          +------------------------------------------------------------------------------------+
```

#### Address Decoding & Data Routing

- To make an array of registers addressable, we introduce two key combinational logic blocks:
  - **Decoder (Address Decoder):** Takes a $k$-bit binary address and activates exactly **one** output line out of $2^k$ rows (the **Word Line**).
  - **Multiplexer (Output Selector):** Takes the data outputs from all registers ($Q_0 \dots Q_3$) and routes the requested register's output onto the Read Bus. We duplicate this block to provide **Port A** and **Port B** concurrently.
- **Write Mechanics:**
  - To write to Register 2, the CPU sets `Write Address = 02` and `LOAD = 1`.
  - The Address Decoder sets $W_2 = 1$ while keeping $W_0, W_1, W_3 = 0$.
  - On the rising clock edge, **only Register 2** updates its D Flip-Flops; all other registers recirculate their internal $Q$ values.
- **Read Mechanics & Read-During-Write Policy:**
  - Reading is purely combinational (asynchronous) and does not wait for a clock tick.
  - Setting `Read Address A = 01` and `Read Address B = 11` routes $Q_1$ directly to `Read Data A` and $Q_3$ directly to `Read Data B` simultaneously.
  - **Read-During-Write:** If the CPU reads and writes to the same register in the same clock cycle, the read ports sample the **existing stored state** ($Q$) before the new data is latched on the clock transition.

#### Python Implementation

In [ ]:
def read_port_4xN(
    slave_state_matrix: list[list[TRANSISTOR_OUTPUT]],  # 4 x N stored outputs
    read_addr: list[TRANSISTOR_OUTPUT]                  # 2-bit address [A1, A0]
) -> list[TRANSISTOR_OUTPUT]:
    '''
    One read port: N 4-to-1 MUXes side by side, sharing the same select lines.
    Bit i of the output is bit i of the register that read_addr points at.

    It reads the stored outputs (Q) and has no clock input, so it returns
    whatever the registers held when it was called.
    '''
    return [
        mux_4to1(q0, q1, q2, q3, sel=read_addr)
        for q0, q1, q2, q3 in zip(*slave_state_matrix)
    ]


def register_file_4xN(
    write_data: list[TRANSISTOR_OUTPUT],
    write_addr: list[TRANSISTOR_OUTPUT],   # 2-bit address [A1, A0]
    read_addr_a: list[TRANSISTOR_OUTPUT],  # 2-bit address [A1, A0]
    read_addr_b: list[TRANSISTOR_OUTPUT],  # 2-bit address [A1, A0]
    write_enable: TRANSISTOR_OUTPUT,       # Global LOAD signal
    enable: TRANSISTOR_OUTPUT,             # System Clock (CLK)
    slave_state_matrix: list[list[TRANSISTOR_OUTPUT]],   # 4 x N stored outputs
    master_state_matrix: list[list[TRANSISTOR_OUTPUT]]   # 4 x N intermediate states
) -> tuple[list[TRANSISTOR_OUTPUT], list[TRANSISTOR_OUTPUT], list[list[TRANSISTOR_OUTPUT]], list[list[TRANSISTOR_OUTPUT]]]:
    '''
    4-Register Addressable Register File (4 words of N-bits each), with two read ports.

    Inputs:
        - write_data: N-bit input data bus
        - write_addr: 2-bit binary write selection
        - read_addr_a, read_addr_b: 2-bit binary read selection for Port A and Port B
        - write_enable: Master LOAD line
        - enable: System clock signal
        - slave_state_matrix: 4xN array holding stored outputs for all 4 registers
        - master_state_matrix: 4xN array holding intermediate states for all 4 registers

    Returns:
        - (read_data_a, read_data_b, new_slave_matrix, new_master_matrix)
        - Both read ports return the values stored before this call. A write to the
          register being read shows up on the next call.
    '''
    # Step 1: Read both ports from the stored state, before any register updates
    read_data_a = read_port_4xN(slave_state_matrix, read_addr_a)
    read_data_b = read_port_4xN(slave_state_matrix, read_addr_b)

    # Step 2: Decode 2-bit Write Address into 4 Word Lines (W0..W3)
    word_lines = decoder_2to4(*write_addr)

    new_slave_matrix = []
    new_master_matrix = []

    # Step 3: Process all 4 Registers in Parallel
    for r in range(4):
        # Register r receives LOAD = 1 ONLY if write_enable is active AND its Word Line is active
        reg_load = cmos_AND(write_enable, word_lines[r])

        # Execute N-Bit Register for Row r
        outputs, _, master_states = n_bit_register(
            inputs=write_data,
            enable=enable,
            load=reg_load,
            slave_vals_prev=slave_state_matrix[r],
            master_vals_prev=master_state_matrix[r]
        )

        new_slave_matrix.append(outputs)
        new_master_matrix.append(master_states)

    return read_data_a, read_data_b, new_slave_matrix, new_master_matrix

### 5. Data RAM

- As we discussed previously, register files must be built according to the service that holds it. We previously at a register file built for an ALU. Now, let's see the build for another type of register file used for Data RAM

- Why do we need to go beyond a Data File in the ALU?
	- The Register File holds 4 words of data (across its 4 internal registers). While 4 words are enough to hold immediate working operands for the ALU, it is far too small to store a program's variables, arrays, or data structures.
	- To store larger datasets, we build Data RAM (Random Access Memory). Architecturally, Data RAM is built using the exact same principles as the Register File. It has registers to store words, an address decoder to pick which row to write, and MUXes to pick which row to read.  
	

- In our CPU, Data RAM expands capacity to 16 words. Because we now have 16 locations, we need a 4-bit address ($2^4 = 16$). To handle 4-bit addresses, we construct two larger logic blocks:
	- 4-to-16 Decoder: Selects 1 of 16 register rows during a write operation (built by combining two 2-to-4 decoders)
	- 16-to-1 MUX: Routes the selected word out to the read bus (built by chaining five 4-to-1 MUXes)

- Unlike the Register File, Data RAM simplifies its access ports:
	- One Shared Address: An instruction either loads from or stores to one RAM address per cycle, so a single 4-bit address line handles both read and write targeting
	- Single Read Port: Because memory operations only fetch one variable at a time, Data RAM only requires one 16-to-1 output MUX (1 Read Port) rather than the dual read ports needed by the Register File

- Note: Just like the Register File, RAM reads are combinational (asynchronous)—the word stored at addr outputs immediately before the clock transition, while writes update on the rising clock edge.   

#### 4-to-16 Decoder & 16-to-1 MUX

```text
4-TO-16 DECODER

 (A3, A2) ──> [ 2-to-4 ] ── H0..H3 ──┐
                                      ├──> Y[4h + l] = AND(H_h, L_l)   (16 outputs)
 (A1, A0) ──> [ 2-to-4 ] ── L0..L3 ──┘

- The high decoder picks a group of 4, the low decoder picks a line inside the group
- Exactly one H and one L are on, so exactly one of the 16 ANDs is on


16-TO-1 MUX

 D0..D3   ──> [ 4-to-1 ] ──┐
 D4..D7   ──> [ 4-to-1 ] ──┤
 D8..D11  ──> [ 4-to-1 ] ──┼──> [ 4-to-1 ] ──> Output
 D12..D15 ──> [ 4-to-1 ] ──┘          ^
                   ^               (S3, S2) picks the group
               (S1, S0) picks the line inside each group
```

In [ ]:
def decoder_4to16(
    a3: TRANSISTOR_OUTPUT,
    a2: TRANSISTOR_OUTPUT,
    a1: TRANSISTOR_OUTPUT,
    a0: TRANSISTOR_OUTPUT
) -> tuple[TRANSISTOR_OUTPUT, ...]:
    '''
    4-to-16 Decoder built from two 2-to-4 decoders.
    Output line 4*h + l is on only when high line h and low line l are both on.
    '''
    high = decoder_2to4(a3, a2)
    low = decoder_2to4(a1, a0)
    return tuple(cmos_AND(h, l) for h in high for l in low)


def mux_16to1(
    data: list[TRANSISTOR_OUTPUT],
    sel: list[TRANSISTOR_OUTPUT]
) -> TRANSISTOR_OUTPUT:
    '''
    16-to-1 MUX built from five 4-to-1 MUXes.

    Inputs:
        - data: 16 data lines [D0 .. D15]
        - sel: 4-bit select [S3, S2, S1, S0] (MSB -> LSB)
    '''
    s3, s2, s1, s0 = sel

    # Stage 1: (S1, S0) picks one line from each group of 4
    group_outputs = [mux_4to1(*data[4 * g: 4 * g + 4], sel=(s1, s0)) for g in range(4)]

    # Stage 2: (S3, S2) picks the group
    return mux_4to1(*group_outputs, sel=(s3, s2))

#### Python Implementation

In [ ]:
def ram_16xN(
    write_data: list[TRANSISTOR_OUTPUT],
    addr: list[TRANSISTOR_OUTPUT],      # 4-bit address [A3, A2, A1, A0]
    write_enable: TRANSISTOR_OUTPUT,
    enable: TRANSISTOR_OUTPUT,          # System Clock (CLK)
    slave_state_matrix: list[list[TRANSISTOR_OUTPUT]],   # 16 x N stored outputs
    master_state_matrix: list[list[TRANSISTOR_OUTPUT]]   # 16 x N intermediate states
) -> tuple[list[TRANSISTOR_OUTPUT], list[list[TRANSISTOR_OUTPUT]], list[list[TRANSISTOR_OUTPUT]]]:
    '''
    16-word RAM, N bits per word. One address for both read and write.

    Inputs:
        - write_data: N-bit word to store
        - addr: 4-bit address
        - write_enable: 1 = store write_data at addr on the rising edge
        - enable: System clock signal
        - slave_state_matrix / master_state_matrix: 16 x N state for all words

    Returns:
        - (read_data, new_slave_matrix, new_master_matrix)
        - read_data is the word stored at addr before this call
    '''
    # Step 1: Read the stored word at addr, before any register updates
    read_data = [mux_16to1(list(column), addr) for column in zip(*slave_state_matrix)]

    # Step 2: Decode the address into 16 Word Lines
    word_lines = decoder_4to16(*addr)

    new_slave_matrix = []
    new_master_matrix = []

    # Step 3: Only the addressed word loads, and only when write_enable is on
    for w in range(16):
        word_load = cmos_AND(write_enable, word_lines[w])

        outputs, _, master_states = n_bit_register(
            inputs=write_data,
            enable=enable,
            load=word_load,
            slave_vals_prev=slave_state_matrix[w],
            master_vals_prev=master_state_matrix[w]
        )

        new_slave_matrix.append(outputs)
        new_master_matrix.append(master_states)

    return read_data, new_slave_matrix, new_master_matrix

### 6. ROM (Instruction Memory)

- The program has to live somewhere the CPU can fetch it from. That is the **ROM** (Read-Only Memory).

- ROM has no write port and no flip-flops. Each bit is a fixed wire connection, made when the chip is built:
    - The address decoder raises one word line
    - Every bit line with a connection to that word line reads 1. Bit lines with no connection read 0
    - So each output bit is an OR of the word lines whose word has a 1 in that position

- Example with 4 words of 4 bits:

```text
              bit3 bit2 bit1 bit0
 word line 0 ──●────┼────●────┼──    word 0 = 1010
 word line 1 ──┼────┼────┼────●──    word 1 = 0001
 word line 2 ──●────●────●────●──    word 2 = 1111
 word line 3 ──┼────┼────┼────┼──    word 3 = 0000
               │    │    │    │
              OR   OR   OR   OR       ● = connection
               ▼    ▼    ▼    ▼
             out3 out2 out1 out0
```

- Here `out3 = OR(word line 0, word line 2)` and `out0 = OR(word line 1, word line 2)`.

- In the simulation, `rom_contents` stands for the wiring. The `if` over it decides which wires exist, not anything the circuit computes at run time. Nothing in the circuit has a write input to it, so a running program cannot change it.

- **Width:** The ROM word width must equal the instruction width, which we choose in notebook 5. So `rom_16xN` works for any width.

- **Harvard vs von Neumann:** If notebook 5 puts code and data in one memory (von Neumann), the program is copied into RAM at power-on instead, and this ROM is not used.

#### Python Implementation

In [ ]:
def rom_16xN(
    rom_contents: list[list[TRANSISTOR_OUTPUT]],
    addr: list[TRANSISTOR_OUTPUT]   # 4-bit address [A3, A2, A1, A0]
) -> list[TRANSISTOR_OUTPUT]:
    '''
    16-word ROM, N bits per word. No clock and no write port.

    Inputs:
        - rom_contents: up to 16 words, each an MSB-first list of N bits. Words past the end read as 0
        - addr: 4-bit address

    Returns:
        - The N-bit word stored at addr
    '''
    assert 1 <= len(rom_contents) <= 16, 'ROM holds 1 to 16 words'

    word_lines = decoder_4to16(*addr)
    n_bits = len(rom_contents[0])

    read_data = []
    for bit in range(n_bits):
        # Bit line: OR of every word line wired to it (every word with a 1 in this position)
        bit_line = GROUND
        for word, word_bits in enumerate(rom_contents):
            if word_bits[bit]:
                bit_line = cmos_OR(bit_line, word_lines[word])
        read_data.append(bit_line)

    return read_data

### 7. Clock Driver

- Having talked so much about clock ticks, we now need an implementation of the CPU clock itself.

- Every stateful function so far takes its state in and hands the new state back. Nothing holds that state between calls, and nothing makes sure the clock goes 0 then 1.

- Recall from the flip-flop section that one clock cycle has two halves:
    - **CLK = 0:** The master latches are open and capture their inputs. The slaves hold, so outputs don't change.
    - **CLK = 1:** The masters close. The slaves open and copy the masters. This is the rising edge, where the stored value changes.

- If we only ever call a flip-flop with CLK = 1, the master never captures anything, and the slave keeps copying whatever stale value the master had. So every cycle must be two calls.

- The clock driver does three things:
    - Holds the state (`slave` and `master` values) of a component between cycles
    - Runs each cycle as a CLK = 0 call followed by a CLK = 1 call
    - Sets the power-on state: every latch starts at GROUND (0)

- The order of events inside one cycle is what the CPU in notebook 5 will rely on:
    1. Read ports return the values stored at the start of the cycle
    2. Combinational logic (e.g. the ALU) computes on those values
    3. The rising edge commits the result into the registers

#### Python Implementation

- `clock_cycle` takes a `step` function, which the caller writes for each component. `step(clk, state)` calls the component with that clock value and returns `(outputs, new_state)`.

- `int_to_bits` and `bits_to_int` convert between Python ints and MSB-first bit lists. They are for building inputs and reading outputs, and are not part of the circuit.

In [ ]:
def int_to_bits(value: int, width: int) -> list[TRANSISTOR_OUTPUT]:
    '''
    Python int -> MSB-first bit list of the given width.
    '''
    return [POWER if (value >> (width - 1 - i)) & 1 else GROUND for i in range(width)]


def bits_to_int(bits: list[TRANSISTOR_OUTPUT]) -> int:
    '''
    MSB-first bit list -> Python int.
    '''
    value = 0
    for bit in bits:
        value = value * 2 + (1 if bit else 0)
    return value


def power_on_state(n_words: int, n_bits: int) -> dict[str, list[list[TRANSISTOR_OUTPUT]]]:
    '''
    State for n_words registers of n_bits each, with every latch at GROUND.
    '''
    # Built row by row, so no two rows are the same list object
    return {
        'slave': [[GROUND] * n_bits for _ in range(n_words)],
        'master': [[GROUND] * n_bits for _ in range(n_words)],
    }


def clock_cycle(step, state):
    '''
    Run one full clock cycle: CLK = 0, then CLK = 1.

    Inputs:
        - step: function (clk, state) -> (outputs, new_state), wrapping one component call
        - state: the component's current state

    Returns:
        - (outputs, new_state): outputs from the CLK = 1 half, and the state after the rising edge
    '''
    _, state = step(GROUND, state)
    outputs, state = step(POWER, state)
    return outputs, state

#### Example: Two Registers in a Chain

- This is the `ADDER --> D Latch --> MULTIPLIER` problem from section 2, with registers.
    - Register A loads a new value every cycle
    - Register B's input is wired to A's output, and B also loads every cycle

- If the flip-flops work, B always holds the value A had one cycle earlier. A value moves exactly one stage per clock cycle.

- Notice that `two_register_chain` passes A's output from the same call straight into B. During CLK = 1, A's output has already changed to the new value, but B's master latch is closed, so B ignores it. B's master captured A's old value during CLK = 0.

In [ ]:
def two_register_chain(clk, state, a_input):
    '''
    Register A loads a_input every cycle. Register B loads A's output every cycle.
    '''
    a_out, _, a_master = n_bit_register(a_input, clk, POWER, state['A']['slave'][0], state['A']['master'][0])
    b_out, _, b_master = n_bit_register(a_out, clk, POWER, state['B']['slave'][0], state['B']['master'][0])

    new_state = {
        'A': {'slave': [a_out], 'master': [a_master]},
        'B': {'slave': [b_out], 'master': [b_master]},
    }
    return (a_out, b_out), new_state


state = {'A': power_on_state(1, 4), 'B': power_on_state(1, 4)}

for cycle, value in enumerate([3, 7, 12, 5]):
    (a_out, b_out), state = clock_cycle(
        lambda clk, st: two_register_chain(clk, st, int_to_bits(value, 4)),
        state
    )
    print(f'cycle {cycle}: input = {value:2d}   A = {bits_to_int(a_out):2d}   B = {bits_to_int(b_out):2d}')

### 8. Tests

- Every later notebook builds on these components. A wrong bit here shows up in notebook 7 as a wrong answer, and is very hard to trace back.

- Each test runs the gate-level component next to a plain Python model and compares them after every clock cycle. Registers are modelled as Python ints, and RAM as a Python list.

- What is covered:
    - **Latches:** SR latch and D latch truth tables, from both starting values
    - **Flip-flop:** holds during CLK = 0, captures on the rising edge, ignores input changes while CLK = 1
    - **Register:** random loads and holds
    - **Register file:** random reads and writes on both ports, read-during-write sees the old value, `write_enable = 0` is ignored
    - **Data RAM:** random reads and writes
    - **ROM:** every address, including addresses past the last word

In [ ]:
import random


def same(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT) -> bool:
    # Inputs may be 0/1 ints or POWER/GROUND; compare as signals
    return bool(a) == bool(b)


def test_latches():
    for prev in (GROUND, POWER):
        assert same(sr_latch(GROUND, GROUND, prev)[0], prev), 'SR latch: hold'
        assert same(sr_latch(POWER, GROUND, prev)[0], POWER), 'SR latch: set'
        assert same(sr_latch(GROUND, POWER, prev)[0], GROUND), 'SR latch: reset'
        for d in (GROUND, POWER):
            assert same(d_latch(d, POWER, prev)[0], d), 'D latch: enabled should pass d'
            assert same(d_latch(d, GROUND, prev)[0], prev), 'D latch: disabled should hold'


def test_flip_flop():
    q, master = GROUND, GROUND
    for d in [POWER, GROUND, POWER, POWER, GROUND]:
        # CLK = 0: master captures d, Q must not change yet
        q_low, _, master = d_flip_flop(d, GROUND, q, master)
        assert same(q_low, q), 'flip-flop: Q changed while CLK = 0'

        # CLK = 1 with the input flipped: Q must take the value captured before the edge
        flipped = GROUND if d else POWER
        q, _, master = d_flip_flop(flipped, POWER, q_low, master)
        assert same(q, d), 'flip-flop: Q did not capture d on the rising edge'


def test_register(n_cycles=30, n_bits=4):
    rng = random.Random(0)
    state = power_on_state(1, n_bits)
    model = 0

    for _ in range(n_cycles):
        value, load = rng.randrange(2 ** n_bits), rng.random() < 0.5

        def step(clk, st):
            out, _, master = n_bit_register(
                int_to_bits(value, n_bits), clk, POWER if load else GROUND, st['slave'][0], st['master'][0]
            )
            return out, {'slave': [out], 'master': [master]}

        out, state = clock_cycle(step, state)
        if load:
            model = value
        assert bits_to_int(out) == model, f'register: got {bits_to_int(out)}, want {model}'


def register_file_cycle(state, write_data, write_addr, addr_a, addr_b, write_enable, n_bits=4):
    '''One clock cycle of register_file_4xN with int inputs.'''
    def step(clk, st):
        a, b, slave, master = register_file_4xN(
            int_to_bits(write_data, n_bits), int_to_bits(write_addr, 2),
            int_to_bits(addr_a, 2), int_to_bits(addr_b, 2),
            POWER if write_enable else GROUND, clk, st['slave'], st['master']
        )
        return (bits_to_int(a), bits_to_int(b)), {'slave': slave, 'master': master}

    return clock_cycle(step, state)


def test_register_file(n_cycles=60):
    rng = random.Random(1)
    state = power_on_state(4, 4)
    model = [0] * 4

    for _ in range(n_cycles):
        data = rng.randrange(16)
        w_addr, a_addr, b_addr = rng.randrange(4), rng.randrange(4), rng.randrange(4)
        write_enable = rng.random() < 0.5

        (a, b), state = register_file_cycle(state, data, w_addr, a_addr, b_addr, write_enable)

        # Read ports see the values stored at the start of the cycle
        assert a == model[a_addr], f'register file port A: R{a_addr} got {a}, want {model[a_addr]}'
        assert b == model[b_addr], f'register file port B: R{b_addr} got {b}, want {model[b_addr]}'
        if write_enable:
            model[w_addr] = data

    assert [bits_to_int(row) for row in state['slave']] == model, 'register file: final state'


def test_read_during_write():
    state = power_on_state(4, 4)
    (_, _), state = register_file_cycle(state, 5, 1, 0, 0, True)            # R1 = 5
    (a, b), state = register_file_cycle(state, 9, 1, 1, 1, True)            # write 9 to R1 while reading R1
    assert (a, b) == (5, 5), f'read-during-write: got {(a, b)}, want old value (5, 5)'
    (a, b), state = register_file_cycle(state, 0, 0, 1, 1, False)
    assert (a, b) == (9, 9), f'read-during-write: next cycle got {(a, b)}, want (9, 9)'


def test_ram(n_cycles=40, n_bits=4):
    rng = random.Random(2)
    state = power_on_state(16, n_bits)
    model = [0] * 16

    for _ in range(n_cycles):
        data, addr = rng.randrange(2 ** n_bits), rng.randrange(16)
        write_enable = rng.random() < 0.5

        def step(clk, st):
            out, slave, master = ram_16xN(
                int_to_bits(data, n_bits), int_to_bits(addr, 4),
                POWER if write_enable else GROUND, clk, st['slave'], st['master']
            )
            return bits_to_int(out), {'slave': slave, 'master': master}

        out, state = clock_cycle(step, state)
        assert out == model[addr], f'RAM: address {addr} got {out}, want {model[addr]}'
        if write_enable:
            model[addr] = data

    assert [bits_to_int(row) for row in state['slave']] == model, 'RAM: final state'


def test_rom():
    rng = random.Random(3)
    for n_words, n_bits in [(16, 4), (16, 8), (5, 6)]:
        words = [rng.randrange(2 ** n_bits) for _ in range(n_words)]
        rom_contents = [int_to_bits(w, n_bits) for w in words]
        for addr in range(16):
            want = words[addr] if addr < n_words else 0
            got = bits_to_int(rom_16xN(rom_contents, int_to_bits(addr, 4)))
            assert got == want, f'ROM ({n_words}x{n_bits}): address {addr} got {got}, want {want}'


for test in [test_latches, test_flip_flop, test_register, test_register_file, test_read_during_write, test_ram, test_rom]:
    test()
    print(f'{test.__name__}: passed')